# OptiKuh — Next-Lactation Milk Yield Forecast

Baseline pipeline: aggregate daily records to **cow (`lom`) + lactation (`lnr`)**, predict **total milk yield of the next lactation** (`mkg_sum` shifted by one within cow).

In [101]:
import pandas as pd
import numpy as np

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# --- constants ---
DISEASE_COLS = ["eu", "kl", "fr", "st", "at", "vo", "pa", "so"]
NUMERIC_SIGNAL_COLS = [
    "fts", "ftsnel", "ftsme", "eb_nel", "eb_me", "h2o",
    "gew", "bcs", "rfd",
    "bhb", "glukose", "nefa", "insulin", "igf1", "adiponektin", "ca", "nsba",
]
GROUP_COLS = ["lom", "lnr"]
TARGET = "target_mkg_sum_next"
ID_COLS = ["lom", "kaldat"]

In [102]:
# --------------------------------------------------
# 1. Load + clean daily records
# --------------------------------------------------

df = pd.read_excel("../data/raw/optikuh.xlsx")

df["datum"] = pd.to_datetime(df["datum"])
df["kaldat"] = pd.to_datetime(df["kaldat"])
df = df.replace("", np.nan)

numeric_cols = NUMERIC_SIGNAL_COLS + [
    "mkg", "alter", "tratag", "trltg", "farm", "rasse", "lnr",
]
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Exclude end-of-lactation / future-looking columns from features (not loaded into X)
# ltg, apltg, trodat, abgdat, gesundheit, kalver, abggrund stay out of aggregation

df = df.sort_values(["lom", "lnr", "datum"]).reset_index(drop=True)
print(f"Daily records: {df.shape}")

Daily records: (492540, 49)


In [103]:
# --------------------------------------------------
# 2. Aggregate to one row per cow + lactation
# --------------------------------------------------

agg_funcs = {"mkg_sum": ("mkg", "sum")}

for col in NUMERIC_SIGNAL_COLS + ["mkg"]:
    for stat in ("mean", "min", "max"):
        agg_funcs[f"{col}_{stat}"] = (col, stat)

for col in ("alter", "tratag", "trltg"):
    agg_funcs[f"{col}_mean"] = (col, "mean")

lact_meta = df.groupby(GROUP_COLS, as_index=False).agg(
    kaldat=("kaldat", "min"),
    farm=("farm", "first"),
    rasse=("rasse", "first"),
)
lact_num = df.groupby(GROUP_COLS, as_index=False).agg(**agg_funcs)

# Disease: one-hot encode text categories, then count per lactation
disease_records = []
for col in DISEASE_COLS:
    mask = df[col].notna()
    if not mask.any():
        continue
    tmp = df.loc[mask, ["lom", "lnr", col]].copy()
    tmp["feature"] = col + "_" + tmp[col].astype(str)
    tmp["val"] = 1
    disease_records.append(tmp[["lom", "lnr", "feature", "val"]])

disease_long = pd.concat(disease_records, ignore_index=True)
disease_agg = (
    disease_long.groupby(["lom", "lnr", "feature"], as_index=False)["val"]
    .sum()
    .pivot(index=["lom", "lnr"], columns="feature", values="val")
    .fillna(0)
    .astype(np.float64)
    .reset_index()
)

lact = lact_meta.merge(lact_num, on=GROUP_COLS, how="left")
lact = lact.merge(disease_agg, on=GROUP_COLS, how="left")

disease_feature_cols = [
    c for c in lact.columns
    if any(c.startswith(f"{d}_") for d in DISEASE_COLS)
]
lact[disease_feature_cols] = lact[disease_feature_cols].fillna(0)

print(f"Lactation-level rows: {lact.shape}")

Lactation-level rows: (3421, 89)


In [104]:
# --------------------------------------------------
# 3. Supervised target: X(lactation n) -> y = mkg_sum(lactation n+1)
# --------------------------------------------------

lact = lact.sort_values(GROUP_COLS)
lact[TARGET] = lact.groupby("lom")["mkg_sum"].shift(-1)

# Drop last lactation per cow (no future target)
model_df = lact.dropna(subset=[TARGET]).copy()
print(f"Supervised rows (last lactation dropped): {model_df.shape}")

Supervised rows (last lactation dropped): (1707, 90)


In [105]:
# --------------------------------------------------
# 4. Time-aware train/test split (by calving date)
# --------------------------------------------------

model_df = model_df.sort_values("kaldat")
split_idx = int(len(model_df) * 0.8)

train = model_df.iloc[:split_idx]
test = model_df.iloc[split_idx:]

drop_from_x = set(ID_COLS + [TARGET, "mkg_sum"])
feature_cols = [c for c in model_df.columns if c not in drop_from_x]

X_train = train[feature_cols].astype(np.float64)
X_test = test[feature_cols].astype(np.float64)
y_train = train[TARGET].astype(np.float64)
y_test = test[TARGET].astype(np.float64)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Calving date range — train: {train['kaldat'].min().date()} to {train['kaldat'].max().date()}")
print(f"Calving date range — test:  {test['kaldat'].min().date()} to {test['kaldat'].max().date()}")

Train: (1365, 86), Test: (342, 86)
Calving date range — train: 2013-03-07 to 2016-02-01
Calving date range — test:  2016-02-02 to 2016-03-28


In [106]:
# --------------------------------------------------
# 5. Dtype check (all features must be numeric)
# --------------------------------------------------

print("Feature dtype counts:\n", X_train.dtypes.value_counts())
bad_dtypes = X_train.select_dtypes(include=["object", "bool"]).columns.tolist()
assert not bad_dtypes, f"Non-numeric feature columns: {bad_dtypes}"
print("All features are float64 — OK")

Feature dtype counts:
 float64    86
Name: count, dtype: int64
All features are float64 — OK


In [107]:
# --------------------------------------------------
# 6. Train baseline model (HistGradientBoostingRegressor)
# --------------------------------------------------

model = HistGradientBoostingRegressor(
    learning_rate=0.05,
    max_depth=8,
    max_iter=500,
    random_state=42,
)

model.fit(X_train, y_train)
print("Model trained.")

Model trained.


In [108]:
# --------------------------------------------------
# 7. Evaluation
# --------------------------------------------------

pred = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, pred))
mae = mean_absolute_error(y_test, pred)
r2 = r2_score(y_test, pred)

print(f"RMSE: {rmse:,.1f} kg")
print(f"MAE:  {mae:,.1f} kg")
print(f"R2:   {r2:.3f}")

RMSE: 3,291.1 kg
MAE:  2,493.5 kg
R2:   0.066


In [109]:
# Quick sanity check: preview supervised dataset
model_df[[*GROUP_COLS, "kaldat", "mkg_sum", TARGET]].head(10)

,lom,lnr,kaldat,mkg_sum,target_mkg_sum_next
1233,DE000536810694,1,2013-03-07,0.0,8603.32
1139,DE000536106967,2,2013-07-09,0.0,4797.61
1033,DE000535428372,4,2013-08-12,0.0,8044.53
1875,DE000769470175,1,2013-08-26,0.0,8499.45
2190,DE000813162367,6,2013-08-28,0.0,2216.67
1716,DE000768607537,6,2013-08-31,0.0,11416.29
1298,DE000537012150,1,2013-09-23,0.0,1196.00
1143,DE000536335247,2,2013-09-28,0.0,9381.54
1126,DE000536106933,2,2013-09-28,0.0,32.19
2832,DE000945671837,1,2013-09-29,0.0,9696.60


## Improvements

Fixes applied to address weak baseline performance (R² ~ 0.06):

1. **Lag features** — prior lactation milk history per cow (ordered by calving date)
2. **Simplified disease encoding** — `disease_count`, `had_disease`, `disease_category_count` (no sparse one-hot)
3. **Data quality filter** — drop lactations with zero recorded milk (incomplete recording window)
4. **Group-aware split** — cows never appear in both train and test (`GroupShuffleSplit` by `lom`)
5. **Feature hygiene** — drop all-NaN / constant columns before training; all features cast to `float64`

In [114]:
from sklearn.model_selection import GroupShuffleSplit

# Narrower, meaningful signal columns (no ltg/apltg/trodat — future/post-calving info)
SIGNAL_COLS = [
    "fts", "ftsnel", "ftsme", "eb_nel", "eb_me",
    "bhb", "glukose", "nefa", "bcs", "gew",
]
LAG_COLS = [
    "prev_lactation_milk",
    "prev_prev_lactation_milk",
    "milk_trend",
    "cow_avg_milk",
]
DISEASE_FEATURE_COLS = ["disease_count", "had_disease", "disease_category_count"]

In [119]:
# --------------------------------------------------
# Step 1–3: Clean aggregation + disease + lag features
# --------------------------------------------------

df_imp = pd.read_excel("data/raw/optikuh.xlsx")
df_imp["datum"] = pd.to_datetime(df_imp["datum"])
df_imp["kaldat"] = pd.to_datetime(df_imp["kaldat"])
df_imp = df_imp.replace("", np.nan)

coerce_cols = SIGNAL_COLS + ["mkg", "alter", "farm", "rasse", "lnr"]
for col in coerce_cols:
    if col in df_imp.columns:
        df_imp[col] = pd.to_numeric(df_imp[col], errors="coerce")

df_imp = df_imp.sort_values(["lom", "lnr", "datum"]).reset_index(drop=True)

# Disease flags at daily level
df_imp["had_disease_row"] = df_imp[DISEASE_COLS].notna().any(axis=1)
df_imp["disease_events_row"] = df_imp[DISEASE_COLS].notna().sum(axis=1)

agg_funcs = {
    "mkg_sum": ("mkg", "sum"),
    "mkg_mean": ("mkg", "mean"),
    "mkg_max": ("mkg", "max"),
    "alter_mean": ("alter", "mean"),
}
for col in SIGNAL_COLS:
    for stat in ("mean", "min", "max"):
        agg_funcs[f"{col}_{stat}"] = (col, stat)

lact_imp = df_imp.groupby(GROUP_COLS, as_index=False).agg(
    kaldat=("kaldat", "min"),
    farm=("farm", "first"),
    rasse=("rasse", "first"),
    **agg_funcs,
)

disease_agg = df_imp.groupby(GROUP_COLS, as_index=False).agg(
    disease_count=("disease_events_row", "sum"),
    had_disease=("had_disease_row", "max"),
)
cat_count = (
    df_imp.groupby(GROUP_COLS)
    .apply(lambda g: g[DISEASE_COLS].notna().any(axis=0).sum(), include_groups=False)
    .reset_index(name="disease_category_count")
)

lact_imp = lact_imp.merge(disease_agg, on=GROUP_COLS).merge(cat_count, on=GROUP_COLS)
lact_imp["had_disease"] = lact_imp["had_disease"].astype(np.float64)

# Order lactations chronologically within cow (not by lnr alone)
lact_imp = lact_imp.sort_values(["lom", "kaldat", "lnr"])

# Lag features — backward-looking only (no future lactation info)
lact_imp["prev_lactation_milk"] = lact_imp.groupby("lom")["mkg_sum"].shift(1)
lact_imp["prev_prev_lactation_milk"] = lact_imp.groupby("lom")["mkg_sum"].shift(2)
lact_imp["milk_trend"] = lact_imp.groupby("lom")["mkg_sum"].diff()
lact_imp["cow_avg_milk"] = lact_imp.groupby("lom")["mkg_sum"].transform(
    lambda s: s.shift(1).expanding().mean()
)

print(f"Lactation rows: {lact_imp.shape}")

Lactation rows: (3421, 46)


In [120]:
# --------------------------------------------------
# Step 4: Supervised dataset  X(lactation n) -> y = mkg_sum(n+1)
# --------------------------------------------------

lact_imp[TARGET] = lact_imp.groupby("lom")["mkg_sum"].shift(-1)

model_df_imp = lact_imp.dropna(subset=[TARGET]).copy()

# Drop lactations with no recorded milk (incomplete observation window → noisy zeros)
n_before = len(model_df_imp)
model_df_imp = model_df_imp[
    (model_df_imp["mkg_sum"] > 0) & (model_df_imp[TARGET] > 0)
].copy()
print(f"Dropped {n_before - len(model_df_imp)} rows with zero milk in current/next lactation")
print(f"Supervised rows: {model_df_imp.shape}")

Dropped 656 rows with zero milk in current/next lactation
Supervised rows: (1051, 47)


In [121]:
# --------------------------------------------------
# Step 5: Group-aware split by cow (no cow in both train & test)
# --------------------------------------------------

drop_from_x = set(ID_COLS + [TARGET])
candidate_features = [c for c in model_df_imp.columns if c not in drop_from_x]

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(model_df_imp, groups=model_df_imp["lom"]))

train_imp = model_df_imp.iloc[train_idx]
test_imp = model_df_imp.iloc[test_idx]

assert set(train_imp["lom"]).isdisjoint(set(test_imp["lom"])), "Cow leakage detected!"

# Drop all-NaN or constant columns (e.g. prev_prev_lactation_milk in small splits)
feature_cols_imp = [
    c for c in candidate_features
    if train_imp[c].notna().any() and train_imp[c].nunique(dropna=True) > 1
]

X_train_imp = train_imp[feature_cols_imp].astype(np.float64)
X_test_imp = test_imp[feature_cols_imp].astype(np.float64)
y_train_imp = train_imp[TARGET].astype(np.float64)
y_test_imp = test_imp[TARGET].astype(np.float64)

print(f"Train: {X_train_imp.shape}, Test: {X_test_imp.shape}")
print(f"Cows — train: {train_imp['lom'].nunique()}, test: {test_imp['lom'].nunique()}")

Train: (834, 44), Test: (217, 44)
Cows — train: 666, test: 167


In [122]:
# --------------------------------------------------
# Dtype check (all features must be numeric)
# --------------------------------------------------

print("Feature dtype counts:\n", X_train_imp.dtypes.value_counts())
bad_dtypes = X_train_imp.select_dtypes(include=["object", "bool"]).columns.tolist()
assert not bad_dtypes, f"Non-numeric feature columns: {bad_dtypes}"
print(f"Using {len(feature_cols_imp)} numeric features — OK")

Feature dtype counts:
 float64    44
Name: count, dtype: int64
Using 44 numeric features — OK


In [123]:
# --------------------------------------------------
# Step 6–7: Train + evaluate improved model
# --------------------------------------------------

model_imp = HistGradientBoostingRegressor(
    learning_rate=0.05,
    max_depth=8,
    max_iter=500,
    random_state=42,
)
model_imp.fit(X_train_imp, y_train_imp)

pred_imp = model_imp.predict(X_test_imp)

rmse_imp = np.sqrt(mean_squared_error(y_test_imp, pred_imp))
mae_imp = mean_absolute_error(y_test_imp, pred_imp)
r2_imp = r2_score(y_test_imp, pred_imp)

print(f"RMSE: {rmse_imp:,.1f} kg")
print(f"MAE:  {mae_imp:,.1f} kg")
print(f"R2:   {r2_imp:.3f}")
print(f"\nBaseline R2 was ~0.06 → improved to {r2_imp:.3f}")

RMSE: 2,555.6 kg
MAE:  1,941.5 kg
R2:   0.453

Baseline R2 was ~0.06 → improved to 0.453


In [124]:
# Preview improved supervised dataset
model_df_imp[[*GROUP_COLS, "kaldat", "mkg_sum", "prev_lactation_milk", "cow_avg_milk", TARGET]].head(10)

,lom,lnr,kaldat,mkg_sum,prev_lactation_milk,cow_avg_milk,target_mkg_sum_next
0,DE000113973296,8,2014-06-02,2475.40,NaN,NaN,7488.66
8,DE000115904006,6,2015-05-07,11281.37,NaN,NaN,6486.87
10,DE000115904018,6,2015-01-13,6913.82,NaN,NaN,14473.42
13,DE000115904082,6,2014-11-25,6652.96,NaN,NaN,14291.50
14,DE000115904082,7,2016-01-25,14291.50,6652.96,6652.96,1665.82
16,DE000115904138,5,2014-07-03,555.10,NaN,NaN,1157.74
21,DE000116244329,5,2014-10-06,2556.16,NaN,NaN,8065.68
23,DE000116244365,5,2015-04-13,11155.47,NaN,NaN,11374.29
27,DE000116244394,5,2015-04-25,10634.99,NaN,NaN,7170.09
30,DE000116594401,5,2015-03-05,10084.02,NaN,NaN,9499.55


In [127]:
X_train

,lnr,farm,rasse,fts_mean,fts_min,fts_max,ftsnel_mean,ftsnel_min,ftsnel_max,ftsme_mean,...,so_Fremdkörper,so_Hauterkrankung,so_Peritonitis,so_Sonstiges,so_traumatisch bedingte Erkrankung,st_Hypocalcämie,st_Ketose,st_Labmagenverlagerung,st_Stoffwechsel Sonstiges,vo_Verdauungsstörung
1233,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1139,2.0,1.0,1.0,12.260000,4.52,37.18,79.674750,29.33,241.30,132.177250,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1033,4.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
1875,1.0,2.0,1.0,14.176667,10.72,17.55,90.724167,68.60,112.31,150.262500,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2190,6.0,8.0,11.0,15.675556,7.62,25.50,88.768889,42.43,141.91,149.305556,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1198,4.0,1.0,1.0,20.754206,6.49,31.80,139.927412,42.59,219.93,229.575235,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1445,2.0,1.0,1.0,25.171198,7.92,42.77,163.832365,49.21,283.18,270.077934,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2610,3.0,4.0,11.0,19.233811,5.21,30.02,128.503567,33.69,203.30,211.734024,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
14,7.0,3.0,1.0,25.427009,6.40,42.02,183.503162,44.95,300.04,299.030000,...,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0
